In [ ]:
## IMPORTS
import os
import shutil

from geneformer import InSilicoPerturber
from geneformer import InSilicoPerturberStats
from geneformer import EmbExtractor

In [ ]:
## SET THE REQUIRED INPUTS

## Path to input data
input_data = "./input_data/example_input_files/cell_classification/disease_classification/human_dcm_hcm_nf.dataset"

## Start and goal (target) states
start_state = 'Fibroblast1'
goal_state = 'Cardiomyocyte1'
alt_states = []

## Perturb-type
perturb_type = 'overexpress'


## Select gene sets for in silico perturbation

# -------------------------------------------
# To select 'All genes', uncomment below line, and comment the section below
# -------------------------------------------

# genes_to_perturb = 'all'

# -------------------------------------------


# -------------------------------------------
# To select specific gene sets, do the following:
# (1) Comment above section
# (2) Uncomment relevant lines from below section corresponding to the dictionary (gene_set) containing genes of interest
# (3) Assign selected specific genes from the relevant gene_set dictionary to a list (genes_to_perturb) that will be used 
# for perturbation

# For e.g. the current setup will select the first two genes from Top-4 genes (i.e. ENSG00000155657 and ENSG00000198626) for 
# perturbation
# -------------------------------------------

# GMTH
# gene_set = {"GATA4": "ENSG00000136574", "MEF2C": "ENSG00000081189",
#                     "TBX5": "ENSG00000089225", "HAND2": "ENSG00000164107"}

# Top-4 genes based on in silico perturbation (overexpression)
gene_set = {"TTN": "ENSG00000155657", "RYR2": "ENSG00000198626",
                    "PDE3A": "ENSG00000172572", "CACNA1C": "ENSG00000151067"}


# Select the genes of interest from gene_set for perturbation
genes_to_perturb = list(gene_set.values())[:2]
# -------------------------------------------


print(f'Following genes will be perturbed: {genes_to_perturb}')

In [ ]:
## Delete old output files and directories and create new ones
output_parent_dir = "./output/reprogramming/"
output_child_dirs = ["in_silico_perturb/", "in_silico_perturb_stats/"]

# Delete the old output directories and their contents
if os.path.exists(output_parent_dir):
    shutil.rmtree(output_parent_dir)


# Create new output directories
for output_child_dir in output_child_dirs:
    if not os.path.exists(os.path.join(output_parent_dir, output_child_dir)):
        os.makedirs(os.path.join(output_parent_dir, output_child_dir))

In [ ]:
## Set the start, goal and alt states (if any) for the run
cell_states_to_model = {"state_key": "cell_type",
                        "start_state": start_state,
                        "goal_state": goal_state,
                        "alt_states": alt_states}


## Filter the input data corresponding to start, goal and alt states (if any)
filter_data_dict = {"cell_type": ["Fibroblast1", "Cardiomyocyte1"] + alt_states}

In [ ]:
## Extract cell state embedding positions from input data
embex = EmbExtractor(model_type="Pretrained",
                     num_classes=0,
                     filter_data=filter_data_dict,
                     emb_mode='cell',
                     max_ncells=500,
                     emb_layer=0,
                     summary_stat="exact_mean",
                     forward_batch_size=16,
                     nproc=4,
                     # change from current default dictionary for 30M model series
                     token_dictionary_file="./Geneformer/geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl")


state_embs_dict = embex.get_state_embs(cell_states_to_model,
                                       "./Geneformer/gf-6L-30M-i2048",
                                       input_data,
                                       output_parent_dir,
                                       "state_embeddings")

In [ ]:
## Perform in silico perturbation of defined set of genes or all genes in sample of cells
isp = InSilicoPerturber(perturb_type="overexpress",
                        perturb_rank_shift=None,
                        genes_to_perturb=genes_to_perturb,
                        combos=0,
                        anchor_gene=None,
                        model_type="Pretrained",
                        num_classes=0,
                        emb_mode="cell",
                        cell_emb_style="mean_pool",
                        filter_data=filter_data_dict,
                        cell_states_to_model=cell_states_to_model,
                        state_embs_dict=state_embs_dict,
                        max_ncells=250,
                        emb_layer=0,
                        forward_batch_size=8,
                        nproc=1, 
                        token_dictionary_file="./Geneformer/geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl")


# Output intermediate files from in silico perturbation
isp.perturb_data("./Geneformer/gf-6L-30M-i2048",
                 input_data,
                 os.path.join(output_parent_dir, output_child_dirs[0]),
                 "perturb_data")

In [ ]:
# Generate in silico perturbation stats
ispstats = InSilicoPerturberStats(
                                mode="goal_state_shift",
                                genes_perturbed=genes_to_perturb,
                                  combos=0,
                                  anchor_gene=None,
                                  cell_states_to_model=cell_states_to_model,
                                  token_dictionary_file="./Geneformer/geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl")


# Extract data from intermediate files and process stats to output in final .csv
ispstats.get_stats(os.path.join(output_parent_dir, output_child_dirs[0]),
                   None,
                   os.path.join(output_parent_dir, output_child_dirs[1]),
                   "isp_stats")